# NB-inject — Fusion modules ON the strong baseline (decisive test)

`unet_drop` (concat U-Net + curriculum) beat LGMF. This tests whether your
fusion contributions **add value on top of that strong backbone**: same MobileNet
encoder + U-Net decoder, same param budget (~6.6M), but naive concat is replaced
by the **gate + FiLM front-end**, and the **consistency loss** is switched on.

The ONLY differences from `unet_drop` are: (1) gated fusion front-end,
(2) consistency loss. Runs the full injection + 2 front-end ablations, 3 seeds.

Decision:
- gated beats unet_drop (0.801 both / 0.598 s1) -> **your modules add value -> paper. (Mamba later = optional upgrade.)**
- gated == unet_drop -> modules are neutral -> pivot to curriculum-as-contribution.

**Attach:** `lgmf_c` (v7), Sen1Floods11, `lgmf-index`. GPU + Internet on. ~2.5h (9 runs).


In [ ]:
import sys, os, glob, json, random, time, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v7"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
try:
    import segmentation_models_pytorch  # noqa
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.baselines import build_gated_unet
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__)
assert lgmf.__version__ >= "7.0.0", "attach lgmf_c v7"
idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))

In [ ]:
SEEDS, EPOCHS = [42, 1, 2], 45
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"; regs = ["both", "s1_only", "s2_only"]

# (name, front-end kwargs, consistency_weight)
RUNS = [
    ("gated_full", dict(film=True,  gate=True),  0.5),   # full injection
    ("gated_nocons", dict(film=True, gate=True), 0.0),   # gate+FiLM, no consistency
    ("gated_nofilm", dict(film=False, gate=True), 0.5),  # gate + consistency, no FiLM
]

def make_cfg(name, seed, cons):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size = EPOCHS, 3e-4, 8
    c.ema_decay, c.loss_type = 0.99, "focal_tversky"
    c.consistency_weight = cons
    c.num_workers = 0
    c.split_dir, c.ckpt_dir = idx, "/kaggle/working/ckpt_inject"
    c.s1_mean, c.s1_std = stats["s1_mean"], stats["s1_std"]
    c.s2_mean, c.s2_std = stats["s2_mean"], stats["s2_std"]
    c.seed, c.run_name = seed, name
    return c

def seed_all(sd):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False

def run_one(cfg, front_kw):
    seed_all(cfg.seed)
    tr = DataLoader(Sen1Floods11(cfg, f"{idx}/train_index.json", augment=True),
                    batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=pin, drop_last=True)
    va = DataLoader(Sen1Floods11(cfg, f"{idx}/val_index.json", augment=False),
                    batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    model = build_gated_unet(pretrained=True, out_ch=8, **front_kw).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type=="cuda") else None
    ema = ModelEMA(model, cfg.ema_decay); em = ema.module
    best, bv = -1.0, None
    for epoch in range(cfg.epochs):
        train_one_epoch(model, tr, opt, cfg, epoch, device, scaler=scaler, ema=ema)
        sched.step()
        val = evaluate(em, va, cfg, device)
        sc = float(np.mean([val[r]["iou"] for r in regs]))
        if sc > best:
            best = sc
            bv = {r: {"iou": val[r]["iou"], "f1": val[r]["f1"],
                      "flood_only_iou": val[r].get("flood_only_iou")} for r in regs}
            D = os.path.join(cfg.ckpt_dir, cfg.run_name, f"seed{cfg.seed}"); os.makedirs(D, exist_ok=True)
            torch.save({"model": model.state_dict(), "ema": ema.module.state_dict(),
                        "gated": {"out_ch": 8, **front_kw},
                        "epoch": epoch, "val": bv, "cfg": dict(cfg.__dict__)},
                       os.path.join(D, "best.pt"))
    return best, bv

## Train (full injection + 2 front-end ablations x 3 seeds)

In [ ]:
results = {name: [] for name, _, _ in RUNS}
t0 = time.time()
for name, front_kw, cons in RUNS:
    for sd in SEEDS:
        D = os.path.join("/kaggle/working/ckpt_inject", name, f"seed{sd}")
        rj = os.path.join(D, "result.json")
        if os.path.exists(rj):
            rec = json.load(open(rj)); best, bv = rec["best"], rec["val"]
            print(f"[skip] {name:12s} seed{sd} mean {best:.3f}")
        else:
            best, bv = run_one(make_cfg(name, sd, cons), front_kw)
            os.makedirs(D, exist_ok=True); json.dump({"best": best, "val": bv}, open(rj,"w"), indent=2, default=str)
            print(f"[done] {name:12s} seed{sd} mean {best:.3f}  both {bv['both']['iou']:.3f} "
                  f"s1 {bv['s1_only']['iou']:.3f} s2 {bv['s2_only']['iou']:.3f}  ({(time.time()-t0)/60:.0f} min)")
        results[name].append(bv)
print(f"\ndone in {(time.time()-t0)/60:.0f} min")

## Verdict vs unet_drop

In [ ]:
def ms(name, reg): a = np.array([r[reg]["iou"] for r in results[name]]); return a.mean(), a.std()
UNET_DROP = {"both": 0.801, "s1_only": 0.598, "s2_only": 0.781}

print("=" * 80)
print(f"{'model':14s} {'both':>13s} {'s1_only':>13s} {'s2_only':>13s}   {'d both':>7s}")
print("-" * 80)
print(f"{'unet_drop':14s} {UNET_DROP['both']:>13.3f} {UNET_DROP['s1_only']:>13.3f} {UNET_DROP['s2_only']:>13.3f}   {'(ref)':>7s}")
for name, _, _ in RUNS:
    b, s1, s2 = ms(name,'both'), ms(name,'s1_only'), ms(name,'s2_only')
    db = b[0] - UNET_DROP['both']
    print(f"{name:14s} {b[0]:.3f}±{b[1]:.3f} {s1[0]:.3f}±{s1[1]:.3f} {s2[0]:.3f}±{s2[1]:.3f}   {db:>+7.3f}")
print("=" * 80)

gb = ms("gated_full", "both")[0]; gs1 = ms("gated_full", "s1_only")[0]
print("\nVERDICT (gated_full vs unet_drop):")
print(f"  both:     {gb:.3f} vs {UNET_DROP['both']:.3f}  ({'WIN' if gb>=UNET_DROP['both'] else 'lose'})")
print(f"  s1_only:  {gs1:.3f} vs {UNET_DROP['s1_only']:.3f}  ({'WIN' if gs1>=UNET_DROP['s1_only'] else 'lose'})")
if gb >= UNET_DROP['both'] - 0.005 and gs1 >= UNET_DROP['s1_only'] - 0.005:
    print("  -> Fusion modules ADD VALUE on the strong backbone. THIS IS THE PAPER MODEL.")
    print("     (Mamba backbone becomes an optional 'even better' v8.)")
else:
    print("  -> Modules don't beat naive concat even on the strong backbone.")
    print("     -> Pivot: the curriculum is the contribution (unet_drop vs unet_naive).")